# Lab02 - Markov Decision Process va Dynamic Programming

**Ho ten:** `<dien ten>`
**MSSV:** `<dien MSSV>`
**Lop:** `<dien lop>`
**GitHub username:** `<dien username>`

Notebook nay minh hoa toan bo 36 bai tap cua Lab02, chia thanh 8 phan:
A. Markov Chain
B. Reward, Return, Discount factor
C. Mo hinh hoa MDP nho
D. Kham pha model FrozenLake
E. Bellman backup va Policy Evaluation
F. Policy Improvement va Policy Iteration
G. Value Iteration
H. Danh gia va so sanh thuat toan

Code chi tiet cua tung bai nam trong `src/bai01.py` ... `src/bai36.py` va
`src/mdp_utils.py`; notebook nay goi lai cac ham do va hien thi ket qua/bieu do.


In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

import mdp_utils as mu

print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)


## Phan A - Markov Chain (Bai 1-6)

In [ ]:
from bai01 import P as weather_P, STATE_NAMES
print(weather_P)
print("Valid:", mu.validate_transition_matrix(weather_P))

p0 = np.array([1.0, 0.0, 0.0])
for t in [1, 2, 5, 10, 50]:
    print(t, mu.state_distribution(p0, weather_P, t))


In [ ]:
# Bai 05-06: mo phong va so sanh voi ly thuyet
rng = np.random.default_rng(42)
n_transitions = 100_000
counts = np.zeros(3)
s = 0
counts[s] += 1
for _ in range(n_transitions):
    s = mu.sample_next_state(s, weather_P, rng)
    counts[s] += 1
empirical = counts / counts.sum()
theoretical = mu.state_distribution(p0, weather_P, 200)
print("Empirical :", np.round(empirical, 4))
print("Theoretical:", np.round(theoretical, 4))


## Phan B - Reward, Return, Discount Factor (Bai 7-11)

In [ ]:
rewards = [0, 0, 0, 0, 10]
gammas = np.linspace(0, 1, 101)
g0_values = [mu.compute_return(rewards, g) for g in gammas]

plt.figure(figsize=(6,4))
plt.plot(gammas, g0_values)
plt.title("Anh huong cua gamma len G_0")
plt.xlabel("gamma"); plt.ylabel("G_0"); plt.grid(True)
plt.savefig("../figures/gamma_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## Phan C - Mo hinh hoa mot MDP nho (Bai 12-15)

In [ ]:
from bai12 import P as small_mdp_P, N_STATES, N_ACTIONS
print(mu.validate_mdp(small_mdp_P, N_STATES, N_ACTIONS))

uniform_policy = np.ones((N_STATES, N_ACTIONS)) / N_ACTIONS
print(uniform_policy)


## Phan D - Kham pha model FrozenLake (Bai 16-20)

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env.reset(seed=42)
print("n_states:", env.observation_space.n, "n_actions:", env.action_space.n)
mu.describe_state(env, 0)


## Phan E - Bellman Backup va Policy Evaluation (Bai 21-25)

In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n
uniform_policy = np.ones((n_states, n_actions)) / n_actions

V, n_iter = mu.policy_evaluation(env, uniform_policy, gamma=0.99, theta=1e-8)
print("Hoi tu sau", n_iter, "iteration")
print(V.reshape(4,4))


## Phan F - Policy Improvement va Policy Iteration (Bai 26-30)

In [ ]:
pi_policy, V_pi, n_pi = mu.policy_iteration(env, gamma=0.99, theta=1e-8)
print("Policy Iteration hoi tu sau", n_pi, "vong lap")
print(V_pi.reshape(4,4))
mu.print_frozenlake_policy(env, pi_policy)


## Phan G - Value Iteration (Bai 31-33)

In [ ]:
V_vi, n_vi, deltas_vi = mu.value_iteration(env, gamma=0.99, theta=1e-8)
optimal_policy = mu.greedy_policy_from_value(env, V_vi, gamma=0.99)
print("Value Iteration hoi tu sau", n_vi, "iteration")
print(V_vi.reshape(4,4))
mu.print_frozenlake_policy(env, optimal_policy)

plt.figure(figsize=(6,4))
plt.plot(range(1, len(deltas_vi)+1), deltas_vi)
plt.yscale("log")
plt.title("Hoi tu cua Value Iteration")
plt.xlabel("Iteration"); plt.ylabel("delta (log)"); plt.grid(True)
plt.savefig("../figures/value_iteration_convergence.png", dpi=150, bbox_inches="tight")
plt.show()


## Phan H - Danh gia va so sanh thuat toan (Bai 34-36)

In [ ]:
stats_vi = mu.evaluate_policy_by_simulation(env, optimal_policy, n_episodes=1000, seed=42)
stats_pi = mu.evaluate_policy_by_simulation(env, pi_policy, n_episodes=1000, seed=42)
print("Value Iteration :", stats_vi)
print("Policy Iteration:", stats_pi)

labels = ["Value Iteration", "Policy Iteration"]
rewards_cmp = [stats_vi["mean_reward"], stats_pi["mean_reward"]]
plt.figure(figsize=(5,4))
plt.bar(labels, rewards_cmp, color=["tab:blue", "tab:green"])
plt.title("So sanh mean reward: VI vs PI")
plt.ylabel("Mean reward"); plt.grid(True, axis="y")
plt.savefig("../figures/algorithm_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

env.close()


## Cau hoi sinh vien phai tra loi

**1. Markov property la gi?**
Tinh chat Markov noi rang trang thai tuong lai chi phu thuoc vao trang thai
hien tai (va action hien tai neu co), khong phu thuoc vao toan bo lich su
truoc do: `P(S_(t+1) | S_t, S_(t-1), ..., S_0) = P(S_(t+1) | S_t)`.

**2. Markov chain khac MDP nhu the nao?**
Markov chain chi co trang thai va xac suat chuyen trang thai, khong co
action va reward. MDP mo rong Markov chain bang cach them action (agent co
the anh huong den qua trinh chuyen trang thai) va reward (tin hieu de danh
gia hanh vi).

**3. Transition probability la gi?**
La xac suat chuyen tu mot state (va action, trong MDP) sang mot state ke
tiep: `p(s' | s, a)` (hoac `p(s' | s)` trong Markov chain thuan).

**4. Vi sao tong transition probability cua mot (state, action) phai bang 1?**
Vi tap hop cac next_state co the la mot bien co day du (chac chan agent se
roi vao mot trong cac state do), nen tong xac suat cua toan bo khong gian
ket qua phai bang 1 (tinh chat co ban cua phan phoi xac suat).

**5. Return khac immediate reward nhu the nao?**
Immediate reward `R_(t+1)` chi la phan thuong nhan duoc ngay sau mot buoc.
Return `G_t` la tong (co chiet khau) cua toan bo reward tu thoi diem t tro
di cho den het episode, phan anh loi ich dai han chu khong chi truoc mat.

**6. Discount factor co vai tro gi?**
`gamma` dieu chinh muc do agent quan tam den reward trong tuong lai so voi
reward hien tai, dong thoi giup return hoi tu (huu han) trong cac bai toan
vong doi vo han.

**7. Khi gamma = 0, agent quan tam dieu gi?**
Agent chi quan tam den reward ngay lap tuc `R_(t+1)`, hoan toan bo qua hau
qua trong tuong lai (agent "can thi", myopic).

**8. Khi gamma gan 1, reward xa trong tuong lai anh huong the nao?**
Reward xa van duoc tinh gan nhu day du (it bi chiet khau), nen agent co xu
huong "nhin xa", chap nhan hy sinh loi ich truoc mat de dat duoc loi ich
lon hon ve sau.

**9. Policy la gi?**
La chien luoc chon action cua agent: anh xa tu state (hoac lich su) sang
action (deterministic) hoac phan phoi xac suat tren action (stochastic).

**10. Deterministic policy khac stochastic policy the nao?**
Deterministic policy luon chon MOT action co dinh cho moi state:
`pi(s) = a`. Stochastic policy tra ve mot phan phoi xac suat tren cac
action: `pi(a|s)`, co the sinh ra action khac nhau moi lan o cung mot state.

**11. V(s) bieu dien dieu gi?**
State-value function: gia tri ky vong cua return khi bat dau tu state `s`
va sau do luon tuan theo policy `pi`.

**12. Q(s,a) bieu dien dieu gi?**
State-action value function: gia tri ky vong cua return khi bat dau tu
state `s`, thuc hien action `a`, roi sau do tuan theo policy `pi`.

**13. Bellman equation co tinh de quy o diem nao?**
V(s) (hoac Q(s,a)) duoc bieu dien qua V (hoac Q) cua chinh cac state ke
tiep: gia tri cua mot state phu thuoc vao gia tri cua cac state khac ma no
co the chuyen den, tao thanh mot he phuong trinh de quy.

**14. Bellman expectation khac Bellman optimality the nao?**
Bellman expectation tinh gia tri theo MOT policy cu the (lay trung binh co
trong so theo `pi(a|s)`). Bellman optimality lay `max` tren tat ca action
thay vi trung binh, tuong ung voi optimal policy.

**15. Dynamic Programming can biet thong tin gi ve moi truong?**
Can biet day du "model" cua MDP: transition probability `p(s'|s,a)` va
reward function `r(s,a,s')` cho moi state-action.

**16. Policy Evaluation dung de lam gi?**
Tinh state-value function `V_pi` cho MOT policy `pi` cho truoc, bang cach
lap Bellman expectation backup den khi hoi tu.

**17. Policy Improvement dung de lam gi?**
Xay dung mot policy moi tot hon (hoac bang) policy cu, bang cach chon
greedy action theo Q(s,a) duoc tinh tu V cua policy cu.

**18. Policy Iteration hoat dong nhu the nao?**
Lap lai hai buoc: (1) Policy Evaluation de tinh V cho policy hien tai, (2)
Policy Improvement de cai thien policy; dung khi policy khong con thay doi
(policy stable).

**19. Value Iteration hoat dong nhu the nao?**
Lap Bellman OPTIMALITY backup truc tiep tren V (khong can policy trung
gian day du): `V(s) <- max_a Q(s,a)`, den khi V hoi tu; sau do trich xuat
policy toi uu bang greedy tu V cuoi cung.

**20. Value Iteration khac Policy Iteration o diem nao?**
Value Iteration gop Policy Evaluation va Improvement thanh MOT buoc (dung
`max` thay vi lap Evaluation day du), thuong can nhieu sweep hon nhung moi
sweep re. Policy Iteration lam Evaluation day du (co the nhieu iteration)
truoc moi lan Improvement, thuong can it vong lap ngoai hon.

**21. Vi sao FrozenLake-v1 thich hop de minh hoa Dynamic Programming?**
Vi FrozenLake co so state va action huu han, nho, va Gymnasium cung cap
day du transition model (`env.unwrapped.P`), dap ung dieu kien tien quyet
cua DP la phai biet truoc model cua moi truong.

**22. is_slippery=True lam model thay doi the nao?**
Moi (state, action) khong con dan den DUY NHAT mot next_state (xac suat 1)
nua, ma co the dan den nhieu next_state khac nhau (huong du dinh va cac
huong vuong goc do truot bang) voi xac suat nho hon 1, doi hoi thuat toan
phai cong theo xac suat tren nhieu transition.

**23. theta anh huong the nao den so iteration?**
`theta` la nguong hoi tu: theta cang nho, dieu kien dung `delta < theta`
cang kho dat duoc, nen thuat toan can nhieu iteration hon de dat do chinh
xac cao hon; theta lon lam thuat toan dung som hon nhung V co the chua
that su hoi tu.

**24. Vi sao can danh gia optimal policy bang simulation?**
Value function tinh boi DP la gia tri LY THUYET (ky vong chinh xac dua
tren model). Simulation (chay nhieu episode thuc te) giup kiem chung ket
qua nay bang thuc nghiem, phat hien loi cai dat, va cho ra cac so lieu de
hieu nhu success rate tren so episode cu the.

**25. Neu khong biet transition model, Dynamic Programming co ap dung
truc tiep duoc khong? Giai thich.**
Khong. Dynamic Programming (Policy Evaluation, Policy Iteration, Value
Iteration nhu trong Lab02) la phuong phap **model-based**: moi buoc backup
deu can `p(s'|s,a)` va `r(s,a,s')` chinh xac. Neu khong biet model, phai
chuyen sang cac phuong phap hoc tu kinh nghiem/episode (model-free) nhu
Monte Carlo, Temporal-Difference Learning, SARSA, Q-Learning - noi thuat
toan uoc luong gia tri thong qua tuong tac thuc te thay vi tinh toan truc
tiep tren model da biet.
